# Вклад групп признаков

In [ ]:
'''
Что уже известно:
- 222 признака дают 0.722 на кросс-валидации и 0.718 на Public;
- время заказов исчерпано: модель на сырых датах не находит ничего сверх формул, правило
  «последний заказ + средний интервал» работает только при 2–3 днях с заказами;
- у других участников результат выше, значит, неиспользованный сигнал есть.

Что делаем здесь:
- делим существующие признаки на группы (время, деньги, скидки и лояльность, товары) и измеряем
  вклад каждой: что даёт добавление группы к времени и что теряется при её удалении из полного набора;
- добавляем четыре новые группы и измеряем прирост каждой к полному набору:
    последние пять заказов «как есть»; счётчики по подкатегориям; траектория лояльности и скидок;
    бренды (уровень цен, популярность, сжатое представление);
- всё считается одной быстрой схемой: LightGBM, 3 фолда, одни и те же фолды для всех наборов.
  Это отсев, а не финальная оценка: что покажет плюс здесь, потом проверяется полным пайплайном.
'''
import os
import re
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb
from IPython.display import display
from scipy import sparse
from scipy.special import ndtr
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 240)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

KAGGLE_DIR = Path("/kaggle/input/competitions/lamoda-cu-repeat-purchase-2026")
DATA_DIR = KAGGLE_DIR if KAGGLE_DIR.exists() else Path(os.environ.get("LAMODA_DATA_DIR", "data"))
OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(os.environ.get("LAMODA_OUT_DIR", "."))

SEED = 42
GAP_DAYS = 14
TARGET_MONTHS = 3
WINDOWS = [7, 14, 30, 60, 90, 180, 365]
N_FOLDS = 3
N_LAST = 5        # сколько последних заказов брать «как есть»
N_SVD = 16
T0 = time.time()


def fast_ap(target, score):
    order = np.argsort(-np.asarray(score), kind="mergesort")
    t = np.asarray(target)[order]
    k = int(np.ceil(len(t) * 0.20))
    top = t[:k]
    return float((np.cumsum(top) / np.arange(1, k + 1) * top).sum() / max(min(k, int(t.sum())), 1))

## Данные и признаки

In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv", parse_dates=["scoring_date"])
test = pd.read_csv(DATA_DIR / "test.csv", parse_dates=["scoring_date"])
sample = pd.read_csv(DATA_DIR / "sample_submission.csv")
orders = pd.read_parquet(DATA_DIR / "orders.parquet")
products = pd.read_parquet(DATA_DIR / "products.parquet")
orders["order_date"] = pd.to_datetime(orders["order_date"])

cust = pd.concat([train.assign(part="train"), test.assign(part="test", target=np.nan)], ignore_index=True)
cust["slice"] = cust["scoring_date"].dt.strftime("%Y-%m-%d")
cust["cutoff"] = cust["scoring_date"] - pd.Timedelta(days=GAP_DAYS)
cust["win_len"] = cust["scoring_date"].map(lambda d: (d + pd.DateOffset(months=TARGET_MONTHS) - d).days)
assert cust["customer_id"].is_unique

lines = orders.merge(products, on="product_id", how="left", validate="m:1").merge(
    cust[["customer_id", "cutoff"]], on="customer_id", how="inner", validate="m:1"
)
lines["d"] = (lines["cutoff"] - lines["order_date"]).dt.days
assert len(lines) == len(orders) and lines["d"].between(0, 732).all()


def clean(name):
    return re.sub(r"\W+", "_", str(name)).strip("_").lower()


def build_features(lines, win_len):
    lines = lines.copy()
    lines["has_disc"] = (lines["discount_percent"] > 0).astype("float32")
    lines["has_loy"] = (lines["loyalty_percent"] > 0).astype("float32")
    lines["weekend"] = (lines["order_date"].dt.dayofweek >= 5).astype("float32")

    # --- уровень заказа, самые свежие заказы идут первыми
    ot = (
        lines.groupby(["customer_id", "order_id"], sort=False)
        .agg(
            d=("d", "first"),
            order_sum=("paid_price", "sum"),
            n_items=("paid_price", "size"),
            disc=("discount_percent", "mean"),
            loy=("loyalty_percent", "max"),
            weekend=("weekend", "first"),
        )
        .reset_index()
        .sort_values(["customer_id", "d", "order_id"])
        .reset_index(drop=True)
    )
    cid = ot["customer_id"]
    f = ot.groupby("customer_id").agg(
        recency=("d", "min"),
        tenure=("d", "max"),
        d_mean=("d", "mean"),
        d_std=("d", "std"),
        n_orders=("order_id", "size"),
        n_items=("n_items", "sum"),
        sum_paid=("order_sum", "sum"),
        order_sum_mean=("order_sum", "mean"),
        order_sum_median=("order_sum", "median"),
        order_sum_max=("order_sum", "max"),
        order_sum_std=("order_sum", "std"),
        items_per_order=("n_items", "mean"),
        items_max=("n_items", "max"),
        order_disc_mean=("disc", "mean"),
        order_loy_mean=("loy", "mean"),
        order_loy_max=("loy", "max"),
        weekend_share=("weekend", "mean"),
    )

    last = ot.drop_duplicates("customer_id").set_index("customer_id")
    first = ot.drop_duplicates("customer_id", keep="last").set_index("customer_id")
    f["last_order_sum"] = last["order_sum"]
    f["last_n_items"] = last["n_items"]
    f["last_disc"] = last["disc"]
    f["last_loy"] = last["loy"]
    f["first_loy"] = first["loy"]
    f["first_order_sum"] = first["order_sum"]
    f["loy_change"] = f["last_loy"] - f["first_loy"]
    f["last_sum_to_mean"] = f["last_order_sum"] / f["order_sum_mean"]

    # --- окна по времени
    for w in WINDOWS:
        m = ot["d"] <= w
        f[f"n_orders_{w}"] = m.groupby(cid).sum()
        f[f"sum_{w}"] = (ot["order_sum"] * m).groupby(cid).sum()
        f[f"n_items_{w}"] = (ot["n_items"] * m).groupby(cid).sum()
    for w in [90, 365]:
        m = ot["d"] <= w
        f[f"loy_max_{w}"] = ot["loy"].where(m).groupby(cid).max()
        f[f"disc_mean_{w}"] = ot["disc"].where(m).groupby(cid).mean()
        f[f"aov_{w}"] = f[f"sum_{w}"] / f[f"n_orders_{w}"].replace(0, np.nan)
    f["n_orders_older_365"] = f["n_orders"] - f["n_orders_365"]

    # --- уровень дня: ритм заказов
    days = ot.groupby(["customer_id", "d"], as_index=False).agg(day_orders=("order_id", "size"), day_sum=("order_sum", "sum"))
    days["gap"] = days.groupby("customer_id")["d"].diff()
    days["rk"] = days.groupby("customer_id").cumcount()
    gd = days.groupby("customer_id")
    f["n_days"] = gd.size()
    f["max_day_orders"] = gd["day_orders"].max()
    f["multi_order_days"] = (days["day_orders"] > 1).groupby(days["customer_id"]).sum()
    f = f.join(gd["gap"].agg(gap_mean="mean", gap_median="median", gap_std="std", gap_min="min", gap_max="max"))
    for k in [1, 2, 4, 9]:
        f[f"d_day_{k + 1}"] = days.loc[days["rk"] == k].set_index("customer_id")["d"]  # давность (k+1)-го с конца дня с заказом
    f["gap_prev"] = days.loc[days["rk"] == 2].set_index("customer_id")["gap"]
    for name, period, limit in [("months_12", 30, 12), ("months_24", 30, 25), ("weeks_13", 7, 13), ("weeks_26", 7, 26)]:
        b = days.assign(b=days["d"] // period)
        f[f"active_{name}"] = b.loc[b["b"] < limit].drop_duplicates(["customer_id", "b"]).groupby("customer_id").size()
    f[[c for c in f.columns if c.startswith("active_")]] = f[[c for c in f.columns if c.startswith("active_")]].fillna(0)

    # --- попадание следующего заказа в целевое окно
    # Время считаем в днях после отсечения: окно — это (GAP_DAYS, GAP_DAYS + win_len].
    r = f["recency"]
    w0, w1 = GAP_DAYS, GAP_DAYS + win_len.reindex(f.index)
    dc = days["customer_id"]
    g, dr = days["gap"], dc.map(r)
    lo, hi = dr + GAP_DAYS, dr + GAP_DAYS + dc.map(win_len)
    n_gaps = g.notna().groupby(dc).sum()
    alive = (g > dr).groupby(dc).sum()  # сколько собственных интервалов клиента длиннее текущей паузы
    f["own_gaps_gt_recency"] = alive
    f["own_share_gaps_gt_recency"] = alive / n_gaps.replace(0, np.nan)
    f["own_p_hit"] = ((g > lo) & (g <= hi)).groupby(dc).sum() / alive.replace(0, np.nan)
    f["own_p_hidden"] = ((g > dr) & (g <= lo)).groupby(dc).sum() / alive.replace(0, np.nan)
    f["own_p_late"] = (g > hi).groupby(dc).sum() / alive.replace(0, np.nan)
    f["own_share_gaps_le_window"] = (g <= dc.map(win_len)).groupby(dc).sum() / n_gaps.replace(0, np.nan)

    wgt = 0.7 ** (days["rk"] - 1).clip(lower=0)
    f["gap_ewm"] = (g * wgt).groupby(dc).sum() / wgt.where(g.notna()).groupby(dc).sum()
    f["gap_last3"] = g.where(days["rk"].between(1, 3)).groupby(dc).mean()
    f["gap_last"] = days.loc[days["rk"] == 1].set_index("customer_id")["gap"]
    f["gap_last_to_median"] = f["gap_last"] / f["gap_median"]
    for name in ["gap_last", "gap_last3", "gap_ewm", "gap_median", "gap_mean", "gap_min", "gap_max"]:
        short = name.replace("gap_", "")
        gg = f[name].where(f[name] > 0)
        t = gg - r  # ожидаемый день следующего заказа; отрицательный — клиент «опаздывает»
        f[f"tnext_{short}"] = t
        f[f"tnext_{short}_dist"] = (w0 - t).clip(lower=0) + (t - w1).clip(lower=0)  # расстояние до окна, 0 — внутри
        if name in ["gap_last", "gap_ewm", "gap_median", "gap_mean"]:
            f[f"cycles_{short}"] = np.floor((r + w1) / gg) - np.floor((r + w0) / gg)  # сколько раз период укладывается в окно
            f[f"tmod_{short}"] = gg - (r % gg)  # до ближайшего кратного периоду дня, если клиент пропускал циклы

    # --- окно в единицах собственного периода и сглаженное попадание
    for name in ["gap_mean", "gap_median", "gap_last", "gap_last3"]:
        short = name.replace("gap_", "")
        gg = f[name].where(f[name] > 0)
        t = gg - r
        f[f"ulo_{short}"] = (r + w0) / gg  # следующий заказ попадает в окно, если ulo < 1 <= uhi
        f[f"uhi_{short}"] = (r + w1) / gg
        for sd in (10, 20):  # день следующего заказа известен с точностью ± sd дней
            f[f"phit{sd}_{short}"] = ndtr((w1 - t) / sd) - ndtr((w0 - t) / sd)
        f[f"phitrel_{short}"] = ndtr((w1 - t) / (0.25 * gg)) - ndtr((w0 - t) / (0.25 * gg))

    # --- дни с заказами по 30-дневным интервалам и в том же окне год и два назад
    mb = pd.crosstab(dc, (days["d"] // 30).clip(upper=24)).reindex(columns=range(24), fill_value=0)
    mb.columns = [f"mb_{i:02d}" for i in mb.columns]
    f = f.join(mb)
    dl = dc.map(win_len)
    for years, tag in [(1, "ya"), (2, "ya2")]:
        start, end = 365 * years - GAP_DAYS - dl, 365 * years - GAP_DAYS - 1
        f[f"{tag}_days"] = ((days["d"] >= start) & (days["d"] <= end)).groupby(dc).sum()
        f[f"{tag}_days_wide"] = ((days["d"] >= start - 30) & (days["d"] <= end + 30)).groupby(dc).sum()

    # --- производные
    f["formula"] = f["n_orders"] / (f["recency"] + 30)
    f["days_formula"] = f["n_days"] / (f["recency"] + 30)
    f["orders_per_30d"] = f["n_orders"] / (f["tenure"] + 30) * 30
    f["recency_to_gap_mean"] = f["recency"] / f["gap_mean"]
    f["recency_to_gap_median"] = f["recency"] / f["gap_median"]
    f["recency_minus_gap_max"] = f["recency"] - f["gap_max"]
    f["gap_cv"] = f["gap_std"] / f["gap_mean"]
    f["recency_to_tenure"] = f["recency"] / (f["tenure"] + 1)
    f["share_orders_30"] = f["n_orders_30"] / f["n_orders"]
    f["share_orders_90"] = f["n_orders_90"] / f["n_orders"]
    f["share_orders_365"] = f["n_orders_365"] / f["n_orders"]
    f["trend_90_vs_365"] = f["n_orders_90"] / (f["n_orders_365"] - f["n_orders_90"] + 1)
    f["trend_30_vs_90"] = f["n_orders_30"] / (f["n_orders_90"] - f["n_orders_30"] + 1)
    f["share_sum_90"] = f["sum_90"] / f["sum_paid"].replace(0, np.nan)

    # --- уровень позиции: цены, скидки, лояльность, состав покупок
    gl = lines.groupby("customer_id")
    f = f.join(
        gl.agg(
            price_mean=("paid_price", "mean"),
            price_median=("paid_price", "median"),
            price_max=("paid_price", "max"),
            price_min=("paid_price", "min"),
            disc_mean=("discount_percent", "mean"),
            disc_max=("discount_percent", "max"),
            disc_share=("has_disc", "mean"),
            loy_mean=("loyalty_percent", "mean"),
            loy_share=("has_loy", "mean"),
            n_loy_values=("loyalty_percent", "nunique"),
            n_products=("product_id", "nunique"),
            n_brands=("brand", "nunique"),
            n_cat=("category", "nunique"),
            n_cat2=("category2", "nunique"),
            n_gender=("gender_product", "nunique"),
        )
    )
    f["repeat_product_share"] = 1 - f["n_products"] / f["n_items"]
    f["brands_per_item"] = f["n_brands"] / f["n_items"]
    f["cat2_per_item"] = f["n_cat2"] / f["n_items"]

    for col, prefix in [("category", "cat"), ("gender_product", "gender")]:
        share = pd.crosstab(lines["customer_id"], lines[col], normalize="index")
        share.columns = [f"{prefix}_share_{clean(c)}" for c in share.columns]
        f = f.join(share)
        recent = lines.loc[lines["d"] <= 90]
        share = pd.crosstab(recent["customer_id"], recent[col], normalize="index")
        share.columns = [f"{prefix}_share90_{clean(c)}" for c in share.columns]
        f = f.join(share)

    for col in ["category2", "brand"]:
        top = (
            lines.groupby(["customer_id", col], observed=True).size().rename("n").reset_index()
            .sort_values(["customer_id", "n", col], ascending=[True, False, True])
            .drop_duplicates("customer_id").set_index("customer_id")
        )
        f[f"top_{col}_share"] = top["n"] / f["n_items"]
        f[f"top_{col}"] = top[col].astype(str)

    # --- ритм отдельно по категориям, подкатегориям и брендам
    for col in ["category", "category2", "brand"]:
        st = (
            lines[["customer_id", col, "d"]].drop_duplicates()
            .groupby(["customer_id", col], observed=True)["d"].agg(n="size", last="min", first="max")
        )
        st = st[st["n"] >= 2].reset_index()
        st["gap"] = (st["first"] - st["last"]) / (st["n"] - 1)
        st["t"] = st["gap"] - st["last"]
        hi = GAP_DAYS + st["customer_id"].map(win_len)
        st["dist"] = (GAP_DAYS - st["t"]).clip(lower=0) + (st["t"] - hi).clip(lower=0)
        st["hit"] = (st["dist"] == 0).astype(float)
        gs = st.groupby("customer_id")
        f[f"ser_{col}_n"] = gs.size()
        f[f"ser_{col}_hit"] = gs["hit"].sum()
        f[[f"ser_{col}_n", f"ser_{col}_hit"]] = f[[f"ser_{col}_n", f"ser_{col}_hit"]].fillna(0)
        f[f"ser_{col}_hit_share"] = gs["hit"].mean()
        f[f"ser_{col}_min_dist"] = gs["dist"].min()
        best = st.sort_values(["customer_id", "dist", "last"]).drop_duplicates("customer_id").set_index("customer_id")
        f[f"ser_{col}_best_t"] = best["t"]
        f[f"ser_{col}_best_gap"] = best["gap"]

    return f.replace([np.inf, -np.inf], np.nan)


feats = build_features(lines, cust.set_index("customer_id")["win_len"])
data = cust.merge(feats, left_on="customer_id", right_index=True, how="left", validate="1:1")
assert data["recency"].notna().all() and len(data) == len(cust)

data = data.drop(columns=["top_brand"])
data["top_category2"] = data["top_category2"].astype("category")
OLD = [c for c in feats.columns if c != "top_brand"]

# --- деление существующих признаков на группы по именам
RULES = [
    ("товары", r"^(n_products|n_brands|n_cat|n_cat2|n_gender|repeat_product_share|brands_per_item|cat2_per_item|cat_share|gender_share|top_|ser_)"),
    ("скидки и лояльность", r"(disc|loy)"),
    ("деньги", r"^(sum_|order_sum|aov_|last_order_sum|first_order_sum|last_sum_to_mean|share_sum|price_|n_items|items_|last_n_items)"),
]
GROUP = {}
for f in OLD:
    GROUP[f] = next((name for name, pat in RULES if re.search(pat, f)), "время")
G = {name: [f for f in OLD if GROUP[f] == name] for name in ["время", "деньги", "скидки и лояльность", "товары"]}
print({k: len(v) for k, v in G.items()})
for k, v in G.items():
    print(f"{k}: {v[:12]} ...")

## Новые группы признаков

In [ ]:
cid = "customer_id"
win = cust.set_index(cid)["win_len"]
lines["full_price"] = ((lines["discount_percent"] == 0) & (lines["loyalty_percent"] == 0)).astype("float32")
cat_code = {c: i for i, c in enumerate(sorted(lines["category"].astype(str).unique()))}
gen_code = {c: i for i, c in enumerate(sorted(lines["gender_product"].astype(str).unique()))}

# --- заказы, самые свежие первыми
main = lines.sort_values([cid, "order_id", "paid_price"], ascending=[True, True, False]).drop_duplicates([cid, "order_id"])
o = (
    lines.groupby([cid, "order_id"], sort=False)
    .agg(d=("d", "first"), order_sum=("paid_price", "sum"), n_items=("paid_price", "size"), price_max=("paid_price", "max"),
         disc=("discount_percent", "mean"), loy=("loyalty_percent", "max"), n_cat2=("category2", "nunique"), full=("full_price", "mean"))
    .reset_index()
)
o["main_cat"] = o["order_id"].map(main.set_index("order_id")["category"].astype(str).map(cat_code))
o["main_gender"] = o["order_id"].map(main.set_index("order_id")["gender_product"].astype(str).map(gen_code))
o = o.sort_values([cid, "d", "order_id"]).reset_index(drop=True)
o["rk"] = o.groupby(cid).cumcount()

# группа 1: последние N_LAST заказов «как есть»
N1 = pd.DataFrame(index=cust[cid])
for col in ["d", "order_sum", "n_items", "price_max", "disc", "loy", "n_cat2", "full", "main_cat", "main_gender"]:
    w = o[o["rk"] < N_LAST].pivot(index=cid, columns="rk", values=col)
    w.columns = [f"o{i + 1}_{col}" for i in w.columns]
    N1 = N1.join(w)

# группа 2: счётчики по подкатегориям и по парам «пол x категория», всего и за 180 дней
lines["gender_category"] = lines["gender_product"].astype(str) + "|" + lines["category"].astype(str)
N2 = pd.DataFrame(index=cust[cid])
for col, prefix in [("category2", "c2"), ("gender_category", "gc")]:
    t = pd.crosstab(lines[cid], lines[col])
    t.columns = [f"{prefix}_{clean(c)}" for c in t.columns]
    N2 = N2.join(t)
recent = lines[lines["d"] <= 180]
t = pd.crosstab(recent[cid], recent["gender_category"])
t.columns = [f"gc180_{clean(c)}" for c in t.columns]
N2 = N2.join(t).fillna(0)

# группа 3: траектория лояльности и скидок
o["loy_prev"] = o.groupby(cid)["loy"].shift(-1)        # лояльность в предыдущем по времени заказе
o["loy_up"] = o["loy"] > o["loy_prev"]
o["loy_down"] = o["loy"] < o["loy_prev"]
go = o.groupby(cid)
N3 = pd.DataFrame(index=cust[cid])
N3["loy_ups"] = go["loy_up"].sum()
N3["loy_downs"] = go["loy_down"].sum()
N3["d_last_loy_up"] = o[o["loy_up"]].groupby(cid)["d"].min()
N3["d_last_loy_down"] = o[o["loy_down"]].groupby(cid)["d"].min()
N3["d_first_loy"] = o[o["loy"] > 0].groupby(cid)["d"].max()
N3["d_last_loy"] = o[o["loy"] > 0].groupby(cid)["d"].min()
N3["loy_last_minus_max"] = go["loy"].first() - go["loy"].max()
N3["loy_last3_mean"] = o[o["rk"] < 3].groupby(cid)["loy"].mean()
N3["loy_orders_share_365"] = (o["loy"] > 0).astype(float).where(o["d"] <= 365).groupby(o[cid]).mean()
N3["disc_last_minus_mean"] = go["disc"].first() - go["disc"].mean()
N3["disc_last3_mean"] = o[o["rk"] < 3].groupby(cid)["disc"].mean()
N3["full_price_share"] = lines.groupby(cid)["full_price"].mean()
N3["full_price_share_180"] = recent.groupby(cid)["full_price"].mean()
N3["deep_disc_share"] = (lines["discount_percent"] >= 50).groupby(lines[cid]).mean()
N3["n_disc_values"] = lines.groupby(cid)["discount_percent"].nunique()
N3["disc_std"] = lines.groupby(cid)["discount_percent"].std()

# группа 4: бренды. Справочные величины по брендам считаются только по заказам до самого раннего отсечения,
# чтобы ни одна строка не получила сведений из заказов позже своей даты.
early = lines[lines["order_date"] <= cust["cutoff"].min()]
brand_price = early.groupby("brand")["paid_price"].median()
brand_pop = np.log1p(early.groupby("brand").size())
brand_buyers = early.groupby("brand")[cid].nunique()
brand_repeat = (early.drop_duplicates([cid, "brand", "order_date"]).groupby(["brand", cid]).size() > 1).groupby("brand").mean()
lines["b_price"] = lines["brand"].map(brand_price)
lines["b_pop"] = lines["brand"].map(brand_pop)
lines["b_repeat"] = lines["brand"].map(brand_repeat.where(brand_buyers >= 30))
gl = lines.groupby(cid)
N4 = pd.DataFrame(index=cust[cid])
N4 = N4.join(gl.agg(b_price_mean=("b_price", "mean"), b_price_max=("b_price", "max"), b_price_std=("b_price", "std"),
                    b_pop_mean=("b_pop", "mean"), b_pop_min=("b_pop", "min"), b_repeat_mean=("b_repeat", "mean"), b_repeat_max=("b_repeat", "max")))
last_main = main[main["order_id"].isin(o.loc[o["rk"] == 0, "order_id"])].set_index(cid)
N4["b_price_last"] = last_main["brand"].map(brand_price)
N4["paid_to_brand_price"] = gl["paid_price"].mean() / N4["b_price_mean"]
brands = pd.Index(sorted(lines["brand"].dropna().astype(str).unique()))
if len(brands) > 2:
    rows = pd.Index(cust[cid])
    def brand_matrix(df):
        cnt = df.groupby([cid, "brand"]).size().reset_index(name="n")
        return sparse.csr_matrix((np.log1p(cnt["n"]), (rows.get_indexer(cnt[cid]), brands.get_indexer(cnt["brand"].astype(str)))), shape=(len(rows), len(brands)))
    k = min(N_SVD, len(brands) - 1)
    svd = TruncatedSVD(n_components=k, random_state=SEED).fit(brand_matrix(early))
    N4 = N4.join(pd.DataFrame(svd.transform(brand_matrix(lines)), index=rows, columns=[f"b_svd_{i}" for i in range(k)]))

NEW = {"последние заказы": N1, "подкатегории": N2, "траектория лояльности": N3, "бренды": N4}
for name, df in NEW.items():
    df = NEW[name] = df.astype("float32")
    data = data.merge(df.reset_index().rename(columns={"index": cid}), on=cid, how="left")
    print(f"{name}: {df.shape[1]} признаков")
N_COLS = {name: list(df.columns) for name, df in NEW.items()}
assert len(data) == len(cust) and data.columns.is_unique

tr = data[data["part"].eq("train")].reset_index(drop=True)
y = tr["target"].astype(int).to_numpy()
sl = tr["slice"].to_numpy()
n_bin = pd.cut(tr["n_days"], [0, 1, 2, 3, 5, 9, 20, 10_000], labels=["1", "2", "3", "4-5", "6-9", "10-20", "21+"])
print(f"подготовка: {(time.time() - T0) / 60:.1f} мин")

## Сравнение наборов

In [ ]:
PARAMS = dict(objective="binary", learning_rate=0.05, n_estimators=3000, num_leaves=31, min_child_samples=100, subsample=0.8,
              subsample_freq=1, colsample_bytree=0.5, reg_lambda=5.0, cat_smooth=50, random_state=SEED, n_jobs=-1, verbose=-1)
folds = list(StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED).split(tr, tr["slice"] + "_" + tr["target"].astype(int).astype(str)))
slices = np.unique(sl)
RUNS = {}


def run(name, features):
    t0 = time.time()
    p, imp = np.zeros(len(tr)), pd.Series(0.0, index=features)
    for a, b in folds:
        m = lgb.LGBMClassifier(**PARAMS).fit(tr[features].iloc[a], y[a], eval_set=[(tr[features].iloc[b], y[b])], callbacks=[lgb.early_stopping(100, verbose=False)])
        p[b] = m.predict_proba(tr[features].iloc[b])[:, 1]
        g = pd.Series(m.booster_.feature_importance("gain"), index=features)
        imp += g / g.sum() / len(folds)
    cells = np.array([[fast_ap(y[b][sl[b] == s], p[b][sl[b] == s]) for s in slices] for _, b in folds])   # фолд x срез
    row = {"признаков": len(features), "AP@20%": float(np.mean([fast_ap(y[sl == s], p[sl == s]) for s in slices]))}
    for nb in n_bin.cat.categories:
        mask = (n_bin == nb).to_numpy()
        if mask.sum() > 50 and 0 < y[mask].mean() < 1:
            row[f"AUC {nb}"] = roc_auc_score(y[mask], p[mask])
    RUNS[name] = dict(row=row, cells=cells, imp=imp, oof=p)
    print(f"{name}: {row['AP@20%']:.4f} ({time.time() - t0:.0f} с)")


ALL = OLD
run("время", G["время"])
run("время + деньги", G["время"] + G["деньги"])
run("время + скидки и лояльность", G["время"] + G["скидки и лояльность"])
run("время + товары", G["время"] + G["товары"])
run("все существующие", ALL)
for name in ["деньги", "скидки и лояльность", "товары"]:
    run(f"все без группы «{name}»", [f for f in ALL if GROUP[f] != name])
for name, colsN in N_COLS.items():
    run(f"все + «{name}»", ALL + colsN)
ALL_NEW = ALL + [c for v in N_COLS.values() for c in v]
run("все + все новые", ALL_NEW)


def table(base):
    out = {}
    for name, r in RUNS.items():
        d = (r["cells"] - RUNS[base]["cells"]).ravel()
        out[name] = {**r["row"], f"разница с «{base}»": d.mean(), "разброс": d.std(), "плюс в парах": f"{int((d > 0).sum())} из {len(d)}"}
    return pd.DataFrame(out).T


print("Что даёт добавление группы к времени:")
display(table("время").loc[["время", "время + деньги", "время + скидки и лояльность", "время + товары", "все существующие"]])
print("Что теряется при удалении группы и что даёт каждая новая группа:")
display(table("все существующие").loc[[k for k in RUNS if k.startswith("все")]])

imp = RUNS["все + все новые"]["imp"]
new_cols = [c for v in N_COLS.values() for c in v]
print("Доля важности по группам в наборе «все + все новые»:")
grp = {**GROUP, **{c: name for name, v in N_COLS.items() for c in v}}
display(imp.groupby(imp.index.map(grp)).sum().sort_values(ascending=False).to_frame("доля важности").T)
print("Самые важные новые признаки:")
display(imp[new_cols].sort_values(ascending=False).head(30).to_frame("доля важности").T)
print(f"Всего времени: {(time.time() - T0) / 60:.1f} мин")